### CELL 1: IMPORTS

In [66]:
import os 
import csv
import json 
import joblib 
import pandas as pd 
import numpy as np 

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder 
from sklearn.preprocessing import StandardScaler 
from sklearn.ensemble import IsolationForest 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import ( 
    classification_report, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    roc_curve, 
    roc_auc_score,
    precision_recall_curve, 
    average_precision_score
) 

print("All Libraries Imported Successfully.")

All Libraries Imported Successfully.


### CELL 2: CONFIGURATION

In [62]:
CSV_PATH = "H:\Machine Learning Models\Intrusion Detection\dataset\Processed\CICIDS_Subset.csv"
LABEL_COL = 'Label'
print(f"Dataset path set to: {CSV_PATH}")

Dataset path set to: H:\Machine Learning Models\Intrusion Detection\dataset\Processed\CICIDS_Subset.csv


### CELL 3: LOAD AND PREPROCESS DATA

In [20]:
CHUNK_SIZE = 50000
SAMPLE_FRAC = 0.3

chunks = []
LABEL_COL = None

try:
    for chunk in pd.read_csv(CSV_PATH,encoding='latin1',chunksize=CHUNK_SIZE,on_bad_lines='skip',engine='python',quoting=csv.QUOTE_NONE):
        chunk.columns = chunk.columns.str.strip()

        if LABEL_COL is None:
            label_candidates = [col for col in chunk.columns if 'label' in col.lower()]
            if not label_candidates:
                raise ValueError("Label column not found in dataset")
            LABEL_COL = label_candidates[0]
            print(f"[INFO] Using label column: {LABEL_COL}")

        chunk = chunk[chunk[LABEL_COL].notna()]
        for col in chunk.columns:
            if col != LABEL_COL:
                chunk[col] = pd.to_numeric(chunk[col], errors='coerce')

        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.fillna(0, inplace=True)
        chunk = chunk.sample(frac=SAMPLE_FRAC, random_state=42)
        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)
    print(f"[INFO] Successfully loaded. Shape: {df.shape}")

    y = df[LABEL_COL]
    X = df.drop(columns=[LABEL_COL])

    if y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)
    print("[INFO] Final Feature Shape:", X.shape)
    del df
except FileNotFoundError:
    print(f"[ERROR] File not found at: {CSV_PATH}")
except Exception as e:
    print(f"[ERROR] Could not read or process CSV: {e}")

[INFO] Using label column: Label
[INFO] Successfully loaded. Shape: (848349, 78)
[INFO] Final Feature Shape: (848349, 77)


In [37]:
print("[INFO] Removing duplicates...")
y = pd.Series(y)
X = X.drop_duplicates()
y = y.loc[X.index]

print("[INFO] New shape:", X.shape)

constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
print("[INFO] Dropping constant columns:", len(constant_cols))
X = X.drop(columns=constant_cols)

for col in X.columns:
    upper = X[col].quantile(0.99)
    lower = X[col].quantile(0.01)
    X[col] = X[col].clip(lower, upper)

[INFO] Removing duplicates...
[INFO] New shape: (787698, 69)
[INFO] Dropping constant columns: 0


In [40]:
df_combined = pd.concat([X, pd.Series(y, name=LABEL_COL)], axis=1)
df_combined = df_combined.drop_duplicates()

y = df_combined[LABEL_COL]
X = df_combined.drop(columns=[LABEL_COL])

print("[INFO] After duplicate removal:", X.shape)

[INFO] After duplicate removal: (785944, 69)


In [41]:
constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
print("[INFO] Dropping constant columns:", len(constant_cols))
X = X.drop(columns=constant_cols)

[INFO] Dropping constant columns: 4


In [42]:
print("\n[INFO] ===== DATA VALIDATION =====")
print(f"[INFO] Samples: {X.shape[0]}, Features: {X.shape[1]}")

label_dist = pd.Series(y).value_counts(normalize=True)
print("\n[INFO] Label Distribution:")
print(label_dist)

if label_dist.iloc[0] > 0.95:
    print("[WARN] Severe class imbalance detected!")

nan_count = X.isna().sum().sum()
inf_count = np.isinf(X.values).sum()
print("\n[INFO] NaN values:", nan_count)
print("[INFO] Infinite values:", inf_count)

if nan_count > 0 or inf_count > 0:
    print("[WARN] Data cleaning incomplete!")

print("\n[INFO] Feature Summary (sample):")
print(X.describe().T[['mean', 'std', 'min', 'max']].head(10))

extreme_cols = [col for col in X.columns if X[col].abs().max() > 1e9]
print("\n[INFO] Extreme value columns:", len(extreme_cols))

constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
print("[INFO] Constant columns:", len(constant_cols))

if len(constant_cols) > 10:
    print("[WARN] Too many useless features!")

dup_count = X.duplicated().sum()
print("[INFO] Duplicate rows:", dup_count)

if dup_count > 0:
    print("[WARN] Dataset contains duplicates!")

if X.shape[1] < 50:
    print("[WARN] Too few features! You may have dropped important columns.")

print("[INFO] ===== VALIDATION COMPLETE =====\n")


[INFO] ===== DATA VALIDATION =====
[INFO] Samples: 785944, Features: 65

[INFO] Label Distribution:
Label
0     0.823531
4     0.066668
2     0.048792
10    0.047650
3     0.003904
7     0.002429
6     0.002082
5     0.002019
11    0.001341
1     0.000739
12    0.000570
14    0.000243
9     0.000019
8     0.000008
13    0.000006
Name: proportion, dtype: float64

[INFO] NaN values: 0
[INFO] Infinite values: 0

[INFO] Feature Summary (sample):
                                     mean           std   min           max
Destination Port             8.419975e+03  1.862516e+04  26.0  6.185803e+04
Flow Duration                1.596526e+07  3.467778e+07   3.0  1.178758e+08
Total Fwd Packets            5.075023e+00  7.671813e+00   1.0  5.000000e+01
Total Backward Packets       4.544295e+00  8.596743e+00   0.0  6.000000e+01
Total Length of Fwd Packets  4.645860e+02  1.490017e+03   0.0  1.160100e+04
Total Length of Bwd Packets  3.227347e+03  9.917032e+03   0.0  7.729539e+04
Fwd Packet Length Max

### CELL 4: TRAIN-TEST-SPLIT

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"[INFO] Data Split Complete.")
print(f"   Total Samples: {len(y)}")
print(f"   Training Samples: {len(y_train)}")
print(f"   Testing Samples: {len(y_test)}")

[INFO] Data Split Complete.
   Total Samples: 785944
   Training Samples: 550160
   Testing Samples: 235784


In [44]:
y_train = (y_train != 0).astype(int)
y_test  = (y_test  != 0).astype(int)

print("[INFO] Binary conversion done.")
print(pd.Series(y_train).value_counts(normalize=True))

[INFO] Binary conversion done.
Label
0    0.823531
1    0.176469
Name: proportion, dtype: float64


In [45]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("[INFO] Scaling complete.")

[INFO] Scaling complete.


In [46]:
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

print("\n[INFO] Before SMOTE:")
print(pd.Series(y_train).value_counts())

sm = SMOTE(sampling_strategy='auto', random_state=42, k_neighbors=3)
X_train, y_train = sm.fit_resample(X_train, y_train)

print("\n[INFO] After SMOTE:")
print(pd.Series(y_train).value_counts())


[INFO] Before SMOTE:
Label
0    453074
1     97086
Name: count, dtype: int64

[INFO] After SMOTE:
Label
0    453074
1    453074
Name: count, dtype: int64


### CELL 5: TRAIN MODELS

In [48]:
#1. Train XGBoostClassifier (Supervised)
print("[INFO] Training XGBoostClassifier...")
xgb = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', n_jobs=1, random_state=42)
xgb.fit(X_train, y_train)
print("[INFO] XGBoost Training Complete...")


#3. Train IsolationForest (UnSupervised)
print("[INFO] Training IsolationForest...")
iso = IsolationForest(n_estimators=100, contamination=0.1, n_jobs=1, random_state=42)
iso.fit(X_train)
print("[INFO] IsolationForest Training Complete...")

[INFO] Training XGBoostClassifier...
[INFO] XGBoost Training Complete...
[INFO] Training IsolationForest...
[INFO] IsolationForest Training Complete...


### CELL 6: EVALUATE MODEL PERFORMANCE

In [49]:
print("[INFO] Evaluating Model on Test Data...")
y_pred = xgb.predict(X_test)

try:
    y_score = xgb.predict_proba(X_test)[:, 1]
except Exception:
    y_score = None
    print("[WARN] Could not get predict_proba scores.")

# Metrics (BINARY)
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print(f"\n[RESULT] Accuracy: {acc * 100:.2f}%")
print(f"[RESULT] Precision: {prec:.4f}")
print(f"[RESULT] Recall: {rec:.4f}")
print(f"[RESULT] F1 Score: {f1:.4f}")

print("\n[RESULT] Confusion Matrix:")
print(pd.DataFrame(cm, index=["Actual Normal", "Actual Attack"], columns=["Pred Normal", "Pred Attack"]))

print("\n[RESULT] Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

[INFO] Evaluating Model on Test Data...

[RESULT] Accuracy: 99.90%
[RESULT] Precision: 0.9952
[RESULT] Recall: 0.9993
[RESULT] F1 Score: 0.9973

[RESULT] Confusion Matrix:
               Pred Normal  Pred Attack
Actual Normal       193975          200
Actual Attack           28        41581

[RESULT] Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    194175
           1       1.00      1.00      1.00     41609

    accuracy                           1.00    235784
   macro avg       1.00      1.00      1.00    235784
weighted avg       1.00      1.00      1.00    235784



In [56]:
# That confirms your test set stayed untouched
original_test_size = len(X_test)
len(X_test) == original_test_size

True

### CELL 7: CALCULATE CURVE DATA (ROC & PR)

In [57]:
print("[INFO] Computing ROC & PR curves...")

# -------------------------
# ROC CURVE
# -------------------------
fpr, tpr, roc_thresholds = roc_curve(y_test, y_score)
roc_auc = roc_auc_score(y_test, y_score)

print("\n[ROC RESULTS]")
print(f"AUC: {roc_auc:.4f}")
print(f"Points: {len(fpr)}")

# -------------------------
# PR CURVE
# -------------------------
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_score)
pr_auc = average_precision_score(y_test, y_score)

print("\n[PR RESULTS]")
print(f"AUC: {pr_auc:.4f}")
print(f"Points: {len(precision)}")

[INFO] Computing ROC & PR curves...

[ROC RESULTS]
AUC: 1.0000
Points: 32328

[PR RESULTS]
AUC: 0.9999
Points: 99864


In [61]:
save_dir = r"H:\Machine Learning Models\Intrusion Detection\dataset\Results"

roc_df = pd.DataFrame({
    "fpr": fpr,
    "tpr": tpr,
    "threshold": roc_thresholds
})

pr_df = pd.DataFrame({
    "precision": precision[:-1],   
    "recall": recall[:-1],
    "threshold": pr_thresholds
})
roc_path = os.path.join(save_dir, "roc_curve.csv")
pr_path  = os.path.join(save_dir, "pr_curve.csv")

roc_df.to_csv(roc_path, index=False)
pr_df.to_csv(pr_path, index=False)

print("\n[INFO] Curve data saved as CSV files.")


[INFO] Curve data saved as CSV files.


In [59]:
print("\n[DEBUG] Score range:", y_score.min(), "→", y_score.max())


[DEBUG] Score range: 1.4546061e-06 → 0.99999785


### CELL 8: SAVE MODELS

In [63]:
output_dir = "models"
os.makedirs(output_dir, exist_ok=True)

try:
    # Save scaler
    joblib.dump(scaler, os.path.join(output_dir, "scaler.joblib"))
    
    # Save supervised model & unsupervised model
    joblib.dump(xgb, os.path.join(output_dir, "xgboost.joblib"))
    joblib.dump(iso, os.path.join(output_dir, "isolation_forest.joblib"))

    print(f"[INFO] All models and scaler saved to '{output_dir}/' directory.")

except Exception as e:
    print(f"[ERROR] Could not save models: {e}")

[INFO] All models and scaler saved to 'models/' directory.


### CELL 9: FEATURE EXTRACTION IMPORTANCE
* Important Only If You Design a Web Interface / Dashboard 

In [64]:
top_features = []
try:
    importances = xgb.feature_importances_
    feature_names = X.columns
    feat_imp = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)[:10]
    top_features = [
        {"feature": f, "importance": float(imp)}
        for f, imp in feat_imp
    ]

    print("\n[INFO] Top 10 Important Features:")
    print("-" * 40)
    for f, imp in feat_imp:
        print(f"{f:<30}: {imp:.5f}")

except Exception as e:
    print(f"[ERROR] Could not extract XGBoost feature importances: {e}")


[INFO] Top 10 Important Features:
----------------------------------------
Average Packet Size           : 0.22999
Bwd Packet Length Std         : 0.20561
Packet Length Variance        : 0.17135
Max Packet Length             : 0.09208
Bwd Header Length             : 0.05019
Fwd Packet Length Mean        : 0.04725
Fwd Packet Length Max         : 0.03538
FIN Flag Count                : 0.01583
Packet Length Std             : 0.01486
Fwd Header Length             : 0.01313


In [65]:
# =========================
# METRICS JSON OUTPUT
# =========================

metrics = {
    "accuracy": float(acc),
    "precision": float(prec),
    "recall": float(rec),
    "f1": float(f1),
    "confusion_matrix": cm.tolist(),
    "top_features": top_features
}

out_dir = os.path.join("static", "data")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "metrics.json")

try:
    with open(out_path, "w") as fh:
        json.dump(metrics, fh, indent=2)
    print(f"[INFO] Metrics written to {out_path}")

except Exception as e:
    print(f"[ERROR] Could not write metrics JSON: {e}")

[INFO] Metrics written to static\data\metrics.json


**Script Ending**